# 01 — Rate Analysis

Cumulative records, response rates, and pause structure for rate-based phenomena:
**Positive Reinforcement**, **Reinforcement Schedules**, **Extinction**,
**Negative Reinforcement**, **Punishment**.

Run `00_data_pipeline` first.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
FIG_DIR  = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})


## Helper functions


In [ ]:
def load_phenomenon(name: str) -> pd.DataFrame:
    path = PROC_DIR / f'{name}.parquet'
    if not path.exists():
        print(f'No data for {name}. Run 00_data_pipeline first.')
        return pd.DataFrame()
    return pd.read_parquet(path)


def response_rate(df: pd.DataFrame,
                  response_code: str,
                  bin_s: float = 10.0) -> pd.DataFrame:
    """Compute responses per bin per session."""
    rows = []
    for (sid, pid, cond), grp in df[df['event_code'] == response_code].groupby(
            ['session_id', 'participant_id', 'condition']):
        max_t = df[(df['session_id'] == sid)]['elapsed_s'].max()
        bins  = np.arange(0, max_t + bin_s, bin_s)
        counts, edges = np.histogram(grp['elapsed_s'].values, bins=bins)
        rows.append(pd.DataFrame({
            'session_id':     sid,
            'participant_id': pid,
            'condition':      cond,
            'bin_start':      edges[:-1],
            'responses':      counts,
            'rate_per_min':   counts / (bin_s / 60),
        }))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def cumulative_record(df: pd.DataFrame,
                      response_code: str,
                      ax,
                      label: str = '',
                      color: str = 'steelblue'):
    """Plot a cumulative record on ax."""
    resp = df[df['event_code'] == response_code].sort_values('elapsed_s')
    if resp.empty:
        return
    t_max = df['elapsed_s'].max()
    t  = np.concatenate([[0], resp['elapsed_s'].values, [t_max]])
    c  = np.concatenate([[0], np.arange(1, len(resp) + 1), [len(resp)]])
    ax.step(t, c, where='post', color=color, label=label, linewidth=1.5)
    ax.set_xlabel('Elapsed time (s)')
    ax.set_ylabel('Cumulative responses')


## 1. Reinforcement Schedules — cumulative records by zone


In [ ]:
sched = load_phenomenon('reinforcement_schedules')

if not sched.empty:
    ZONE_COLORS = {'FR': '#e63946', 'VR': '#f4a261', 'FI': '#2a9d8f', 'VI': '#457b9d'}
    ZONE_ORDER  = ['FR', 'VR', 'FI', 'VI']

    # Add zone column from condition field (condition stores zone label in this phenomenon)
    sched['zone'] = sched['condition'].str.extract(r'(FR|VR|FI|VI)', expand=False)

    sessions = sched['session_id'].unique()
    fig, axes = plt.subplots(len(sessions), 4, figsize=(16, 4 * len(sessions)),
                             sharex='col', sharey='col')
    axes = np.atleast_2d(axes)

    for row_i, sid in enumerate(sessions):
        sess = sched[sched['session_id'] == sid]
        for col_i, zone in enumerate(ZONE_ORDER):
            ax = axes[row_i, col_i]
            zone_data = sess[sess['zone'] == zone]
            if not zone_data.empty:
                cumulative_record(zone_data, 'RESPONSE', ax,
                                  label=zone, color=ZONE_COLORS[zone])
                # Mark SR+ deliveries as tick marks
                sr = zone_data[zone_data['event_code'] == 'SR_PLUS']
                if not sr.empty:
                    cr_at_sr = [zone_data[zone_data['elapsed_s'] <= t]['event_code']
                                .eq('RESPONSE').sum() for t in sr['elapsed_s']]
                    ax.vlines(sr['elapsed_s'], cr_at_sr,
                              [v + 0.5 for v in cr_at_sr],
                              color='gold', linewidth=2, label='SR+')
            ax.set_title(f'{zone}  (session {sid})')
            ax.legend(fontsize=8)

    fig.suptitle('Cumulative Records by Schedule Zone', fontsize=14, y=1.01)
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'schedules_cumulative_records.png', bbox_inches='tight')
    plt.show()


## 2. Post-reinforcement pause by schedule


In [ ]:
if not sched.empty:
    # Post-SR pause = elapsed_s of first RESPONSE after SR_PLUS minus elapsed_s of SR_PLUS
    pause_rows = []
    for (sid, zone), grp in sched.groupby(['session_id', 'zone']):
        events = grp.sort_values('elapsed_s')[['elapsed_s', 'event_code']]
        for i, row in events.iterrows():
            if row['event_code'] == 'SR_PLUS':
                sr_t = row['elapsed_s']
                next_resp = events[(events.index > i) &
                                   (events['event_code'] == 'RESPONSE')]
                if not next_resp.empty:
                    pause = next_resp.iloc[0]['elapsed_s'] - sr_t
                    pause_rows.append({'session_id': sid, 'zone': zone, 'pause_s': pause})

    if pause_rows:
        pause_df = pd.DataFrame(pause_rows)
        fig, ax = plt.subplots(figsize=(8, 4))
        for zone in ZONE_ORDER:
            z = pause_df[pause_df['zone'] == zone]['pause_s']
            if not z.empty:
                ax.bar(zone, z.mean(), yerr=z.sem(), capsize=4,
                       color=ZONE_COLORS.get(zone, 'gray'), alpha=0.85)
        ax.set_ylabel('Post-SR pause (s, mean ± SEM)')
        ax.set_title('Post-Reinforcement Pause by Schedule')
        fig.savefig(FIG_DIR / 'schedules_psr_pause.png', bbox_inches='tight')
        plt.show()
        display(pause_df.groupby('zone')['pause_s']
                        .agg(['mean','std','count'])
                        .rename(columns={'mean':'mean_s','std':'sd_s','count':'n'}))


## 3. Extinction — response rate and burst across sessions


In [ ]:
ext = load_phenomenon('extinction')

if not ext.empty:
    SESSION_LABELS = {1: 'Baseline (CRF)', 2: 'Extinction',
                      3: 'Spont. Recovery', 4: 'Re-extinction'}

    # Assume session number is stored in condition column or must be inferred
    # from session order per participant
    ext = ext.sort_values(['participant_id', 'datetime'])
    ext['session_num'] = ext.groupby('participant_id')['session_id'] \
                            .transform(lambda x: pd.factorize(x)[0] + 1)

    # Response rate per 30-second bin per session
    rate_df = []
    for (pid, snum), grp in ext.groupby(['participant_id', 'session_num']):
        resp = grp[grp['event_code'] == 'RESPONSE']
        max_t = grp['elapsed_s'].max()
        bins  = np.arange(0, max_t + 30, 30)
        if len(bins) < 2:
            continue
        counts, edges = np.histogram(resp['elapsed_s'].values, bins=bins)
        for i, c in enumerate(counts):
            rate_df.append({'participant_id': pid, 'session_num': snum,
                            'session_label': SESSION_LABELS.get(snum, str(snum)),
                            'bin_start': edges[i], 'responses': c,
                            'rate_per_min': c / 0.5})
    rate_df = pd.DataFrame(rate_df)

    if not rate_df.empty:
        colors = ['#2a9d8f','#e63946','#f4a261','#457b9d']
        sessions = sorted(rate_df['session_num'].unique())
        fig, axes = plt.subplots(1, len(sessions), figsize=(5 * len(sessions), 4),
                                 sharey=True)
        for ax, (snum, color) in zip(np.atleast_1d(axes), zip(sessions, colors)):
            s = rate_df[rate_df['session_num'] == snum]
            for pid, pgrp in s.groupby('participant_id'):
                ax.plot(pgrp['bin_start'], pgrp['rate_per_min'],
                        marker='o', ms=4, label=pid, alpha=0.7)
            ax.set_title(SESSION_LABELS.get(snum, str(snum)))
            ax.set_xlabel('Time in session (s)')
            if ax is axes[0] if hasattr(axes, '__iter__') else axes:
                ax.set_ylabel('Responses / min')
            ax.legend(fontsize=8)
        fig.suptitle('Extinction — Response Rate Across Sessions', fontsize=13)
        plt.tight_layout()
        fig.savefig(FIG_DIR / 'extinction_rate_by_session.png', bbox_inches='tight')
        plt.show()


## 4. Extinction burst detection


In [ ]:
if not ext.empty:
    burst_df = ext[ext['event_code'] == 'EXT_BURST']
    if burst_df.empty:
        print('No extinction bursts coded in this dataset.')
    else:
        print(f'Extinction bursts: {len(burst_df)}')
        display(burst_df[['participant_id', 'session_num', 'elapsed_s']]
                .sort_values(['participant_id', 'elapsed_s']))

        # Latency to first burst per session
        first_burst = (burst_df.groupby(['participant_id', 'session_num'])['elapsed_s']
                               .min().rename('latency_to_burst_s').reset_index())
        display(first_burst)


## 5. Punishment — path allocation across ABAB phases


In [ ]:
pun = load_phenomenon('punishment')

if not pun.empty:
    PHASE_ORDER = ['A1', 'B1', 'A2', 'B2']

    choices = pun[pun['event_code'].isin(['TARGET_RESP', 'SUPPRESSED'])].copy()
    choices['phase'] = choices['condition']  # condition stores phase in this phenomenon

    alloc = (choices.groupby(['participant_id', 'phase', 'event_code'])
                    .size().rename('count').reset_index())
    alloc['pct_punished'] = alloc.groupby(['participant_id', 'phase'])['count'] \
                                  .transform(lambda x: x / x.sum() * 100)
    punished = alloc[alloc['event_code'] == 'TARGET_RESP']

    if not punished.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        for pid, pgrp in punished.groupby('participant_id'):
            pgrp = pgrp.set_index('phase').reindex(PHASE_ORDER)
            ax.plot(PHASE_ORDER, pgrp['pct_punished'],
                    marker='o', label=pid)
        ax.axhline(50, ls='--', color='gray', alpha=0.5, label='50% (indifference)')
        ax.set_ylabel('% choices on punished (left) path')
        ax.set_title('Punishment — Path Allocation Across ABAB Phases')
        ax.set_ylim(0, 100)
        ax.legend()
        fig.savefig(FIG_DIR / 'punishment_abab_allocation.png', bbox_inches='tight')
        plt.show()
